# Assignment 1: Vector Database Creation and Retrieval
## Day 6 Session 2 - RAG Fundamentals

**OBJECTIVE:** Create a vector database from a folder of documents and implement basic retrieval functionality.

**LEARNING GOALS:**
- Understand document loading with SimpleDirectoryReader
- Learn vector store setup with LanceDB
- Implement vector index creation
- Perform semantic search and retrieval

**DATASET:** Use the data folder in `Day_6/session_2/data/` which contains multiple file types

**INSTRUCTIONS:**
1. Complete each function by replacing the TODO comments with actual implementation
2. Run each cell after completing the function to test it
3. The answers can be found in the existing notebooks in the `llamaindex_rag/` folder


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -r /content/drive/MyDrive/ai-accelerator-C5-main/Day_6/session_2/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 13.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 15.1 MB/s eta 0:00:00
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
INFO: pip is looking at multiple versions of llama-index-llms-openai-like to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-index-llms-openai-like to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of llama-index-

In [4]:
try:
    import llama_index.vector_stores.lancedb
    print("✅ 'llama-index-vector-stores-lancedb' is installed.")
except ImportError:
    print("❌ 'llama-index-vector-stores-lancedb' is NOT installed.")

✅ 'llama-index-vector-stores-lancedb' is installed.


In [ ]:
# Import required libraries
import os
from pathlib import Path
from typing import List
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext, Settings
from llama_index.vector_stores.lancedb import LanceDBVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

print("✅ Libraries imported successfully! Great Job!")

In [ ]:
# Configure LlamaIndex Settings (Using OpenRouter - No OpenAI API Key needed)
from google.colab import userdata

def setup_llamaindex_settings():
    """
    Configure LlamaIndex with local embeddings and OpenRouter for LLM.
    This assignment focuses on vector database operations, so we'll use local embeddings only.
    """
    # Check for OpenRouter API key (for future use, not needed for this basic assignment)
    api_key = userdata.get('OPENROUTER_API_KEY')
    if api_key:
        print(f"ℹ️  OPENROUTER_API_KEY found: {api_key[:4]}...{api_key[-4:]}")
    else:
        print("ℹ️  OPENROUTER_API_KEY not found - that's OK for this assignment!")
        print("   This assignment only uses local embeddings for vector operations." \
        "If we plan to use OpenRouter later, we'll add your API key to Colab Secrets.")

    # Configure local embeddings (no API key required)
    Settings.embed_model = HuggingFaceEmbedding(
        model_name="BAAI/bge-small-en-v1.5",
        trust_remote_code=True
    )

    print("✅ LlamaIndex configured with local embeddings")
    print("   Using BAAI/bge-small-en-v1.5 for document embeddings")

# Setup the configuration
setup_llamaindex_settings()

ℹ️  OPENROUTER_API_KEY found: sk-o...be5c
✅ LlamaIndex configured with local embeddings
   Using BAAI/bge-small-en-v1.5 for document embeddings


## 1. Document Loading Function

Complete the function below to load documents from a folder using `SimpleDirectoryReader`.

**Note:** This assignment uses local embeddings only - no OpenAI API key required! We're configured to use OpenRouter for future LLM operations.


In [ ]:
def load_documents_from_folder(folder_path: str):
    """
    Load documents from a folder using SimpleDirectoryReader.

    Args:
        folder_path (str): Path to the folder containing documents

    Returns:
        List of documents loaded from the folder
    """
    # Initialize SimpleDirectoryReader: This object is responsible for scanning the specified folder recursively.
    # `input_dir` points to the directory where my documents are located.
    # `recursive=True` ensures that documents in subfolders within `folder_path` are also loaded.
    reader = SimpleDirectoryReader(input_dir=folder_path, recursive=True)

    # Load the data: This method reads all supported document types (like PDFs, TXT, MD, etc.)
    # from the specified directory and converts them into a list of LlamaIndex Document objects.
    documents = reader.load_data()

    # Return the loaded documents. Each document object contains the text content
    # and potentially metadata extracted from the original file.
    return documents

# Test the function after you complete it
# Define the path to the data folder for testing purposes.
test_folder = "/content/drive/MyDrive/ai-accelerator-C5-main/Day_6/session_2/data"
# Call the function to load documents from the test folder.
documents = load_documents_from_folder(test_folder)
# Print the total number of documents successfully loaded to confirm functionality.
print(f"Loaded {len(documents)} documents")

In [12]:
if documents:
    print(f"Viewing the first document (ID: {documents[0].id_})")
    print("\n--- Document Text (first 500 characters) ---")
    print(documents[0].text[:500])
    print("\n--- End of Document Text ---")
    print("\n--- Document Metadata ---")
    print(documents[0].metadata)
    if 'file_path' in documents[0].metadata:
        print(f"\n--- Document File Path ---")
        print(documents[0].metadata['file_path'])
    else:
        print("\n--- Document File Path ---")
        print("File path not found in metadata.")
else:
    print("No documents loaded to display.")

Viewing the first document (ID: c6ac9330-7549-45bd-ae22-e434e03ba8c0)

--- Document Text (first 500 characters) ---
A Comprehensive Survey of AI Agent Frameworks
and Their Applications in Financial Services
Satyadhar Joshi
Independent
Alumnus, International MBA, Bar-Ilan University, Israel
satyadhar.joshi@gmail.com
Abstract—This paper surveys the landscape of AI agent
frameworks, highlights their core features and differences, and
explores their applications in financial services. We synthesize
insights from recent industry reports, academic research, and
technical blog posts, focusing on frameworks such as C

--- End of Document Text ---

--- Document Metadata ---
{'page_label': '1', 'file_name': 'AI_Agent_Frameworks.pdf', 'file_path': '/content/drive/MyDrive/ai-accelerator-C5-main/Day_6/session_2/data/AI_Agent_Frameworks.pdf', 'file_type': 'application/pdf', 'file_size': 360523, 'creation_date': '2026-03-29', 'last_modified_date': '2026-03-28'}

--- Document File Path ---
/content/dr

## 2. Vector Store Creation Function

Complete the function below to create a LanceDB vector store.


In [ ]:
def create_vector_store(db_path: str = "./vectordb", table_name: str = "documents"):
    """
    Create a LanceDB vector store for storing document embeddings.
    We have already created documentns out of the raw data, now create the vector store to store the embeddings of those documents.

    TODO: Complete this function to create and configure a LanceDB vector store.
    HINT: Use LanceDBVectorStore with uri and table_name parameters

    Args:
        db_path (str): Path where the vector database will be stored
        table_name (str): Name of the table in the vector database

    Returns:
        LanceDBVectorStore: Configured vector store
    """
    # Create the directory if it doesn't exist
    Path(db_path).mkdir(parents=True, exist_ok=True)

    # Create vector store
    vector_store = LanceDBVectorStore(uri=db_path, table_name=table_name)

    return vector_store

# Test the function after you complete it
# Define the path and table name for testing
test_db_path = "./assignment_vectordb"
test_table_name = "documents"

vector_store = create_vector_store(test_db_path, test_table_name)
print(f"Vector store created: {vector_store is not None}")

if vector_store:
    print(f"LanceDB directory: {test_db_path}")
    print(f"Table name: {test_table_name}")
    # Access the underlying LanceDB table to get its schema
    # Note: The table might not have a schema until documents are actually added
    # but we can try to inspect it if it exists.
    if hasattr(vector_store, '_table') and vector_store._table:
        print("\n--- LanceDB Table Schema ---")
        print(vector_store._table.schema)
    else:
        print("\nLanceDB table not yet created or has no schema (will be created when documents are indexed).")


Vector store created: True
LanceDB directory: ./assignment_vectordb
Table name: documents

--- LanceDB Table Schema ---
id: string
doc_id: string
vector: fixed_size_list<item: float>[384]
  child 0, item: float
text: string
metadata: struct<_node_content: string, _node_type: string, creation_date: string, doc_id: string, document_id: string, file_name: string, file_path: string, file_size: int64, file_type: string, last_modified_date: string, page_label: string, ref_doc_id: string>
  child 0, _node_content: string
  child 1, _node_type: string
  child 2, creation_date: string
  child 3, doc_id: string
  child 4, document_id: string
  child 5, file_name: string
  child 6, file_path: string
  child 7, file_size: int64
  child 8, file_type: string
  child 9, last_modified_date: string
  child 10, page_label: string
  child 11, ref_doc_id: string


In [19]:
import os

relative_path = './assignment_vectordb'
absolute_path = os.path.abspath(relative_path)
print(f"The LanceDB directory is located at: {absolute_path}")

The LanceDB directory is located at: /content/assignment_vectordb


## 3. Vector Index Creation Function

Complete the function below to create a vector index from documents.


In [ ]:
def create_vector_index(documents: List, vector_store):
    """
    Create a vector index from documents using the provided vector store.

    Args:
        documents: List of documents to index
        vector_store: LanceDB vector store to use for storage

    Returns:
        VectorStoreIndex: The created vector index
    """
    # Create storage context: This component manages where the index's data (like embeddings and metadata)
    # will be stored. We pass our LanceDB vector_store to it, indicating that LanceDB should be used
    # as the backend for vector storage.
    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    """ Create index from documents: This is the core step where the vector index is built.
     LlamaIndex takes the loaded documents, processes them (e.g., chunks them if necessary),
     generates embeddings for each chunk using the configured embedding model (BAAI/bge-small-en-v1.5),
     and then stores these embeddings and associated text in the LanceDB vector store
     managed by the storage_context."""
    index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)

    return index

# Test the function after it is completed (will only work after previous functions are completed)
if documents and vector_store:
    index = create_vector_index(documents, vector_store)
    print(f"Vector index created: {index is not None}")
    # Add verification for the LanceDB table content
    if index and hasattr(vector_store, '_table') and vector_store._table:
        try:
            # Use test_table_name which is defined in the previous cell and accessible
            print(f"LanceDB table '{test_table_name}' contains {vector_store._table.to_pandas().shape[0]} records.")
        except Exception as e:
            print(f"Could not count records in LanceDB table: {e}")
    else:
        print("LanceDB table object not available for verification.")
else:
    print("Complete previous functions first to test this one")

Vector index created: True
LanceDB table 'documents' contains 165 records.


In [ ]:
import lancedb
import pandas as pd

# Re-use the path and table name from previous cells
db = lancedb.connect(test_db_path)
table = db.open_table(test_table_name)

# Convert the LanceDB table to a pandas DataFrame and display the first 5 rows
# This will show the 'vector' (embedding), 'text', 'metadata', and other columns
print(f"Displaying the first 5 records from LanceDB table '{test_table_name}':")
display(table.to_pandas().head())


Displaying the first 5 records from LanceDB table 'documents':


,id,doc_id,vector,text,metadata
0,2f4c36c9-f379-40cc-9849-cba694fd129a,4c63a5af-c53e-425e-952e-7c2d56c68f15,"[-0.053176872, -0.035969984, -0.0653414, -0.00...",A Comprehensive Survey of AI Agent Frameworks\...,"{'_node_content': '{""id_"": ""2f4c36c9-f379-40cc..."
1,6b6d22ee-3b05-48ab-b7b4-fc42654e052c,4c63a5af-c53e-425e-952e-7c2d56c68f15,"[-0.049613602, -0.038200915, -0.049676284, -0....","However, several important considerations need...","{'_node_content': '{""id_"": ""6b6d22ee-3b05-48ab..."
2,a002d6d3-53ab-464b-8538-e87628f8f970,33f8435f-5d58-4c4f-a535-8a22305b836d,"[-0.05077551, -0.06461373, -0.07861938, 0.0016...",Several platforms and frameworks have emerged ...,"{'_node_content': '{""id_"": ""a002d6d3-53ab-464b..."
3,c5ff9f3d-9b85-44df-b0dc-b5c941c39f51,33f8435f-5d58-4c4f-a535-8a22305b836d,"[-0.055867326, -0.049481425, -0.07699835, -0.0...",II. AI A GENT FRAMEWORKS : A C OMPARATIVE ANAL...,"{'_node_content': '{""id_"": ""c5ff9f3d-9b85-44df..."
4,55cb1287-e42a-4966-9aee-f2d3d1aceeac,1f438a66-ec0e-49ca-91b0-4d26b723f4dd,"[-0.03581131, -0.033892654, -0.07731759, 0.022...",F . Comparison and Suitability for Finance\nTh...,"{'_node_content': '{""id_"": ""55cb1287-e42a-4966..."


## 4. Document Search Function

Complete the function below to search for relevant documents using the vector index.


In [ ]:
def search_documents(index, query: str, top_k: int = 3):
    """
    Search for relevant documents using the vector index.

    Args:
        index: Vector index to search
        query (str): Search query
        top_k (int): Number of top results to return

    Returns:
        List of retrieved document nodes
    """
    print(f"Searching for documents related to: '{query}' with top_k={top_k}")

    # Create a retriever from the index.
    # The retriever is responsible for fetching relevant documents based on the query.
    # `similarity_top_k` specifies how many of the most similar documents to retrieve.
    retriever = index.as_retriever(similarity_top_k=top_k)
    print("Retriever created successfully.")

    # Retrieve documents for the given query.
    # This performs a semantic search, embedding the query and finding document embeddings
    # that are most similar in the vector space.
    results = retriever.retrieve(query)
    print(f"Retrieved {len(results)} results for the query.")

    return results

# Test the function after we complete it (will only work after all previous functions are completed)
if 'index' in locals() and index is not None:
    test_query = "What are AI agents?"
    results = search_documents(index, test_query, top_k=2)
    print(f"Found {len(results)} results for query: '{test_query}'")
    for i, result in enumerate(results, 1):
        # Accessing `text` attribute of the NodeWithScore object
        text_preview = result.text[:100] if hasattr(result, 'text') else 'No text available'
        # Accessing `score` attribute of the NodeWithScore object
        score = f" (Score: {result.score:.4f})" if hasattr(result, 'score') else ""
        print(f"Result {i}: {text_preview}...{score}")
else:
    print("Complete all previous functions first to test this one")


Searching for documents related to: 'What are AI agents?' with top_k=2
Retriever created successfully.
Retrieved 2 results for the query.
Found 2 results for query: 'What are AI agents?'
Result 1: task. In single agent patterns there is no feedback mechanism from other AI agents; however, there m... (Score: 0.6206)
Result 2: task. In single agent patterns there is no feedback mechanism from other AI agents; however, there m... (Score: 0.6206)


## 5. Final Test - Complete Pipeline

Once you've completed all the functions above, run this cell to test the complete pipeline with multiple search queries.


In [24]:
# Final test of the complete pipeline
print("🚀 Testing Complete Vector Database Pipeline")
print("=" * 50)

# Re-run the complete pipeline to ensure everything works
# Update data_folder to use an absolute path for consistent execution
data_folder = "/content/drive/MyDrive/ai-accelerator-C5-main/Day_6/session_2/data"
vector_db_path = "./assignment_vectordb"

# Step 1: Load documents
print("\n📂 Step 1: Loading documents...")
documents = load_documents_from_folder(data_folder)
print(f"   Loaded {len(documents)} documents")

# Step 2: Create vector store
print("\n🗄️ Step 2: Creating vector store...")
vector_store = create_vector_store(vector_db_path)
print("   Vector store status:", "✅ Created" if vector_store else "❌ Failed")

# Step 3: Create vector index
print("\n🔗 Step 3: Creating vector index...")
if documents and vector_store:
    index = create_vector_index(documents, vector_store)
    print("   Index status:", "✅ Created" if index else "❌ Failed")
else:
    index = None
    print("   ❌ Cannot create index - missing documents or vector store")

# Step 4: Test multiple search queries
print("\n🔍 Step 4: Testing search functionality...")
if index:
    search_queries = [
        "What are AI agents?",
        "How to evaluate agent performance?",
        "Italian recipes and cooking",
        "Financial analysis and investment"
    ]

    for query in search_queries:
        print(f"\n   🔎 Query: '{query}'")
        results = search_documents(index, query, top_k=2)

        if results:
            for i, result in enumerate(results, 1):
                text_preview = result.text[:100] if hasattr(result, 'text') else "No text available"
                score = f" (Score: {result.score:.4f})" if hasattr(result, 'score') else ""
                print(f"      {i}. {text_preview}...{score}")
        else:
            print("      No results found")
else:
    print("   ❌ Cannot test search - index not created")

print("\n" + "=" * 50)
print("🎯 Assignment Status:")
print(f"   Documents loaded: {'✅' if documents else '❌'}")
print(f"   Vector store created: {'✅' if vector_store else '❌'}")
print(f"   Index created: {'✅' if index else '❌'}")
print(f"   Search working: {'✅' if index else '❌'}")

if documents and vector_store and index:
    print("\n🎉 Congratulations! You've successfully completed the assignment!")
    print("   You've built a complete vector database with search functionality!")
else:
    print("\n📝 Please complete the TODO functions above to finish the assignment.")


🚀 Testing Complete Vector Database Pipeline

📂 Step 1: Loading documents...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


   Loaded 42 documents

🗄️ Step 2: Creating vector store...
   Vector store status: ✅ Created

🔗 Step 3: Creating vector index...
   Index status: ✅ Created

🔍 Step 4: Testing search functionality...

   🔎 Query: 'What are AI agents?'
Searching for documents related to: 'What are AI agents?' with top_k=2
Retriever created successfully.
Retrieved 2 results for the query.
      1. task. In single agent patterns there is no feedback mechanism from other AI agents; however, there m... (Score: 0.6206)
      2. task. In single agent patterns there is no feedback mechanism from other AI agents; however, there m... (Score: 0.6206)

   🔎 Query: 'How to evaluate agent performance?'
Searching for documents related to: 'How to evaluate agent performance?' with top_k=2
Retriever created successfully.
Retrieved 2 results for the query.
      1. steps, but the answers are limited to Yes/No responses [7]. As the industry continues to pivot towar... (Score: 0.6749)
      2. steps, but the answers are l